In [1]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
GPU available: True
GPU: Tesla T4


In [2]:
%pip install -U pandas numpy scikit-learn
%pip install -U transformers datasets accelerate safetensors
%pip install -U matplotlib

  Using cached transformers-5.23.0-py3-none-any.whl.metadata (31 kB)
  Using cached datasets-5.2.0-py3-none-any.whl.metadata (24 kB)
  Using cached pyarrow-26.0.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached transformers-5.23.0-py3-none-any.whl (13.3 MB)
Using cached datasets-5.2.0-py3-none-any.whl (608 kB)
Using cached pyarrow-26.0.1-cp313-cp313-manylinux_2_28_x86_64.whl (55.0 MB)
  Attempting uninstall: datasets
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [pyarrow]    WARNING: Ignoring invalid distribution ~yarrow (/usr/local/lib/python3.13/dist-packages)
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [transformers]    WARNING: Ignoring invalid distribution ~yarrow (/usr/local/lib/python3.13/dist-packages)
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1

In [3]:
!git clone https://github.com/Break-Through-Tech/Accenture-1D-contract-review-challenge.git

Cloning into 'Accenture-1D-contract-review-challenge'...
remote: Enumerating objects: 561, done.
remote: Counting objects: 100% (344/344), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 561 (delta 177), reused 269 (delta 125), pack-reused 217 (from 2)
Receiving objects: 100% (561/561), 82.12 MiB | 11.46 MiB/s, done.
Resolving deltas: 100% (251/251), done.
Updating files: 100% (118/118), done.


In [4]:
from pathlib import Path

ROOT = Path(
    "/content/Accenture-1D-contract-review-challenge"
)

print("Repository exists:", ROOT.exists())

Repository exists: True


In [5]:
for file in ROOT.rglob("*"):
    if file.is_file() and (
        "train" in file.name.lower()
        or "validation" in file.name.lower()
        or "split" in file.name.lower()
    ):
        print(file.relative_to(ROOT))

notebooks/create_split.ipynb
data/splits/split_manifest.csv
data/splits/split_verification.json
data/splits/train.json
data/splits/validation.json
data/gate3/validation_chunk_labels_768_512.csv
data/gate3/train_chunks_768_512.csv
data/gate3/train_chunk_labels_768_512.csv
data/gate3/validation_chunks_768_512.csv
data/cuad/cleaned_train_df.csv
data/cuad/train_separate_questions.json
data/risk/category_training_examples.csv
data/risk/risk_training_examples.jsonl
reports/tf_idf_baseline/validation_margins.csv
reports/figures/training_validation_category_support.png
reports/figures/training_document_lengths.png
reports/keyword_baseline/validation_metrics.csv
reports/keyword_baseline/validation_threshold_search.csv
reports/keyword_baseline/validation_predictions.csv
reports/keyword_baseline/train_sanity_metrics.csv
reports/keyword_baseline/validation_match_evidence.csv
reports/gate3_evaluation/validation_summary.csv
reports/gate3_evaluation/validation_error_groups.csv
reports/gate3_evaluatio

In [6]:
import pandas as pd

val_chunks = pd.read_csv(
    ROOT / "data/gate3/validation_chunks_768_512.csv"
)

val_labels = pd.read_csv(
    ROOT / "data/gate3/validation_chunk_labels_768_512.csv"
)

print("Validation chunks:", len(val_chunks))
print("Category pairs:", len(val_labels))

assert len(val_chunks) == 2481
assert len(val_labels) == 24810

assert val_chunks["chunk_id"].is_unique
assert not val_labels.duplicated(
    ["chunk_id", "category"]
).any()

assert (
    val_labels.groupby("chunk_id")["category"]
    .nunique()
    .eq(10)
    .all()
)

print("Validation structure verified!")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Validation chunks: 2481
Category pairs: 24810
Validation structure verified!


In [7]:
import json

candidate_dir = (
    ROOT
    / "configs/Accenture_October_Transformer"
    / "selected_transformer_candidate_a"
)

with open(
    candidate_dir / "category_order.json",
    encoding="utf-8"
) as f:
    category_data = json.load(f)

if isinstance(category_data, list):
    categories = category_data
else:
    categories = category_data["category_order"]

assert len(categories) == 10
assert len(set(categories)) == 10

print(categories)

['Anti-Assignment', 'Change Of Control', 'Expiration Date', 'Renewal Term', 'License Grant', 'Non-Transferable License', 'Exclusivity', 'Non-Compete', 'Cap On Liability', 'Uncapped Liability']


In [8]:
from pathlib import Path

print("Searching for model checkpoints...\n")

for file in ROOT.rglob("*"):
    if file.is_file() and file.suffix in [
        ".safetensors", ".bin", ".pt", ".pth"
    ]:
        print(file)

Searching for model checkpoints...



In [9]:
print("Candidate directory:", candidate_dir)

for file in candidate_dir.iterdir():
    print(file.name)

Candidate directory: /content/Accenture-1D-contract-review-challenge/configs/Accenture_October_Transformer/selected_transformer_candidate_a
category_order.json
tokenizer_config.json
metadata.json
config.json
tokenizer.json
file_hashes.json
thresholds.json


In [10]:
print("Training-related files:\n")

for file in ROOT.rglob("*"):
    if file.is_file() and any(
        word in file.name.lower()
        for word in ["train", "split", "chunk"]
    ):
        print(file.relative_to(ROOT))

Training-related files:

notebooks/gate3_chunking.ipynb
notebooks/create_split.ipynb
data/splits/split_manifest.csv
data/splits/split_verification.json
data/splits/train.json
data/gate3/validation_chunk_labels_768_512.csv
data/gate3/train_chunks_768_512.csv
data/gate3/train_chunk_labels_768_512.csv
data/gate3/validation_chunks_768_512.csv
data/cuad/cleaned_train_df.csv
data/cuad/train_separate_questions.json
data/risk/category_training_examples.csv
data/risk/risk_training_examples.jsonl
reports/figures/training_validation_category_support.png
reports/figures/training_document_lengths.png
reports/keyword_baseline/train_sanity_metrics.csv
src/keyword_baseline/score_chunks.py
src/keyword_baseline/train_sanity.py


In [11]:
%whos

Variable        Type             Data/Info
------------------------------------------
Path            type             <class 'pathlib._local.Path'>
ROOT            PosixPath        /content/Accenture-1D-contract-review-challenge
candidate_dir   PosixPath        /content/Accenture-1D-con<...>d_transformer_candidate_a
categories      list             n=10
category_data   list             n=10
f               TextIOWrapper    <_io.TextIOWrapper name='<...>ode='r' encoding='utf-8'>
file            PosixPath        /content/Accenture-1D-con<...>_baseline/train_sanity.py
json            module           <module 'json' from '/usr<...>on3.13/json/__init__.py'>
pd              module           <module 'pandas' from '/u<...>ages/pandas/__init__.py'>
sys             module           <module 'sys' (built-in)>
torch           module           <module 'torch' from '/us<...>kages/torch/__init__.py'>
val_chunks      DataFrame                                 <...>n[2481 rows x 15 columns]
val_labels  

In [12]:
import pandas as pd

train_chunks = pd.read_csv(
    ROOT / "data/gate3/train_chunks_768_512.csv"
)

train_labels = pd.read_csv(
    ROOT / "data/gate3/train_chunk_labels_768_512.csv"
)

print("Training chunks:", train_chunks.shape)
print("Training labels:", train_labels.shape)

print("\nTraining chunk columns:")
print(train_chunks.columns.tolist())

print("\nTraining label columns:")
print(train_labels.columns.tolist())

print("\nFirst training chunk:")
display(train_chunks.head(2))

print("\nFirst training labels:")
display(train_labels.head(5))

Training chunks: (9765, 15)
Training labels: (97650, 8)

Training chunk columns:
['chunk_id', 'title', 'config', 'config_version', 'chunk_size', 'overlap', 'step_size', 'chunk_index', 'token_start', 'token_end', 'token_count', 'char_start', 'char_end', 'text', 'text_hash']

Training label columns:
['chunk_id', 'title', 'config', 'chunk_index', 'category', 'target', 'mask', 'label']

First training chunk:


,chunk_id,title,config,config_version,chunk_size,overlap,step_size,chunk_index,token_start,token_end,token_count,char_start,char_end,text,text_hash
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,gate3_chunking_v1,768,512,256,0,0,768,768,0,7031,EXHIBIT 10.6\n\n ...,3e3e2a51f53a74b71fa3f5f9c585ce14c8d648480b2864...
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,gate3_chunking_v1,768,512,256,1,256,1024,768,1922,8970,"of Products in the Market, subject to the ter...",f8581902e070805a39bfb582228bfa03ac891057caabe4...



First training labels:


,chunk_id,title,config,chunk_index,category,target,mask,label
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Anti-Assignment,0,0,negative
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Change Of Control,0,0,negative
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Expiration Date,1,0,positive
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,Renewal Term,1,0,positive
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,768_512,0,License Grant,1,0,positive


In [13]:
import json

with open(candidate_dir / "config.json") as f:
    config = json.load(f)

print("Model type:", config.get("model_type"))
print("Architecture:", config.get("architectures"))
print("Number of labels:", config.get("num_labels"))

with open(candidate_dir / "metadata.json") as f:
    metadata = json.load(f)

print("\nTraining metadata:")
print(json.dumps(metadata, indent=2)[:4000])

Model type: modernbert
Architecture: ['ModernBertForSequenceClassification']
Number of labels: None

Training metadata:
{
  "selected_candidate": "A - Unweighted",
  "base_model": "answerdotai/ModernBERT-base",
  "best_epoch": 1,
  "best_validation_loss": 0.0801,
  "validation_macro_f1": 0.7637635286636082,
  "validation_micro_f1": 0.808282208588957,
  "validation_precision": 0.8325434439178515,
  "validation_recall": 0.7853949329359166,
  "validation_tp": 1054,
  "validation_fp": 212,
  "validation_fn": 288,
  "validation_tn": 23092,
  "max_length": 4096,
  "train_batch_size": 2,
  "gradient_accumulation_steps": 4,
  "learning_rate": 3e-05,
  "weight_decay": 0.01,
  "epochs": 3,
  "seed": 42,
  "chunk_size": 768,
  "chunk_overlap": 512,
  "num_categories": 10,
  "selection_note": "Candidate A was selected because it achieved higher micro-F1 and precision, fewer false positives, and lower validation loss. Candidate B achieved slightly higher macro-F1 and recall, but did not consistentl

In [14]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [15]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

checkpoints = list(
    drive_root.rglob("model.safetensors")
)

print("Checkpoints found:", len(checkpoints))

for checkpoint in checkpoints:
    print(checkpoint)

Checkpoints found: 1
/content/drive/MyDrive/model.safetensors


In [16]:
from pathlib import Path
import hashlib
import json

checkpoint = Path("/content/drive/MyDrive/model.safetensors")

assert checkpoint.is_file(), "Checkpoint not found"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

actual_hash = sha256_file(checkpoint)

print("Checkpoint:", checkpoint)
print("Size (MB):", round(checkpoint.stat().st_size / 1e6, 2))
print("SHA256:", actual_hash)

with open(candidate_dir / "file_hashes.json") as f:
    hashes = json.load(f)

print("\nRecorded hashes:")
print(json.dumps(hashes, indent=2)[:4000])

Checkpoint: /content/drive/MyDrive/model.safetensors
Size (MB): 598.46
SHA256: cc8ff6a049fe46ccfde88992e758d7dae7f06b6871c4f327cf8606e72e27aa52

Recorded hashes:
{
  "category_order.json": "38e598b7053d693d87a7ce6988c7e61d7109cd44a4b1d044967b3a7d36180480",
  "config.json": "8e9c74478ae1457d41054cecf76e139da790e1d7073085f2d4bfa018eb70c730",
  "metadata.json": "b54e632c6f30ae719077ad7f6ee790095878a5729203b2b8a72dc870cc82dd57",
  "model.safetensors": "cc8ff6a049fe46ccfde88992e758d7dae7f06b6871c4f327cf8606e72e27aa52",
  "thresholds.json": "e1c789b2ae4eaed09475cbf9491920c373e50ddee78180d82fbc9a356738c2b5",
  "tokenizer.json": "0fc88ec706f80b2eb15174d04fa7377948ca3f07065985deec1e28d3fb1cda9d",
  "tokenizer_config.json": "fb54f027372062b2ca52282efb04d178a8b57167a00cd8f4e816515823a2c016"
}


In [17]:
import torch
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from safetensors.torch import load_file

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# Load the original model configuration
config = AutoConfig.from_pretrained(
    str(candidate_dir),
    local_files_only=True
)

print("Model labels:", config.num_labels)

assert config.num_labels == len(categories), (
    f"Expected {len(categories)} outputs, "
    f"found {config.num_labels}"
)

# Load the original tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    str(candidate_dir),
    local_files_only=True
)

# Construct architecture without downloading generic
# pretrained weights or randomly substituting a checkpoint.
model = AutoModelForSequenceClassification.from_config(config)

# Load the fine-tuned weights
state_dict = load_file(str(checkpoint), device="cpu")

model.load_state_dict(state_dict, strict=True)

model = model.to(device)
model.eval()

print("Candidate A loaded successfully!")

Using device: cuda
Model labels: 10
Candidate A loaded successfully!


In [18]:
print("Validation chunks:", len(val_chunks))
print("Validation labels:", len(val_labels))

assert len(val_chunks) == 2481
assert len(val_labels) == 24810

assert val_chunks["chunk_id"].is_unique

assert not val_labels.duplicated(
    ["chunk_id", "category"]
).any()

print("First text sample:")
print(str(val_chunks.iloc[0]["text"])[:500])

Validation chunks: 2481
Validation labels: 24810
First text sample:
Exhibit 10.26    CONFIDENTIAL TREATMENT HAS BEEN REQUESTED AS TO CERTAIN PORTIONS OF THIS DOCUMENT. EACH SUCH PORTION,  WHICH HAS BEEN OMITTED HEREIN AND REPLACED WITH AN ASTERISK [*], HAS BEEN FILED SEPARATELY WITH THE  SECURITIES AND EXCHANGE COMMISSION.     PROMOTION AND DISTRIBUTION AGREEMENT     This Promotion and Distribution Agreement including all exhibits (collectively referred to as the "Agreement"), effective as of 1 August 2011 (the  "Effective Date"), is made by and between Whitesmo


In [19]:
texts = val_chunks["text"].iloc[:2].fillna("").tolist()

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=4096,
    return_tensors="pt"
)

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

with torch.inference_mode():
    outputs = model(**inputs)
    probabilities = torch.sigmoid(outputs.logits)

print("Output shape:", probabilities.shape)
print("Probabilities:")
print(probabilities.cpu().numpy())

Output shape: torch.Size([2, 10])
Probabilities:
[[2.28434365e-04 8.79221261e-05 5.41759073e-04 2.06100929e-04
  1.15122634e-03 9.70477995e-04 9.31186951e-04 1.20986144e-04
  6.25810528e-04 6.53198629e-04]
 [2.93726305e-04 9.24160122e-05 2.15561842e-04 1.60296026e-04
  9.41909442e-04 1.27550121e-03 6.88054424e-04 1.61166085e-04
  7.39765703e-04 4.56891110e-04]]


In [20]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Current device:", next(model.parameters()).device)

CUDA available: True
Current device: cuda:0


In [21]:
import json

with open(candidate_dir / "thresholds.json") as f:
    thresholds_data = json.load(f)

print(json.dumps(thresholds_data, indent=2))

{
  "Anti-Assignment": 0.405786395072937,
  "Change Of Control": 0.3152312636375427,
  "Expiration Date": 0.027410054579377174,
  "Renewal Term": 0.02888680249452591,
  "License Grant": 0.4142394959926605,
  "Non-Transferable License": 0.8023521900177002,
  "Exclusivity": 0.09771333634853363,
  "Non-Compete": 0.08559418469667435,
  "Cap On Liability": 0.5162952542304993,
  "Uncapped Liability": 0.7782624959945679
}


In [22]:
print("Transformer category order:")

for i, category in enumerate(categories):
    print(i, category)

print("\nValidation categories:")
print(sorted(val_labels["category"].unique()))

Transformer category order:
0 Anti-Assignment
1 Change Of Control
2 Expiration Date
3 Renewal Term
4 License Grant
5 Non-Transferable License
6 Exclusivity
7 Non-Compete
8 Cap On Liability
9 Uncapped Liability

Validation categories:
['Anti-Assignment', 'Cap On Liability', 'Change Of Control', 'Exclusivity', 'Expiration Date', 'License Grant', 'Non-Compete', 'Non-Transferable License', 'Renewal Term', 'Uncapped Liability']


In [23]:
import numpy as np
import torch
from tqdm.auto import tqdm

model.eval()

all_probabilities = []

batch_size = 2

for start in tqdm(range(0, len(val_chunks), batch_size)):

    batch_texts = (
        val_chunks["text"]
        .iloc[start:start + batch_size]
        .fillna("")
        .tolist()
    )

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=4096,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)

    all_probabilities.append(
        probs.cpu().numpy()
    )

probabilities = np.concatenate(
    all_probabilities,
    axis=0
)

print("Prediction shape:", probabilities.shape)

assert probabilities.shape == (2481, 10)

print("Full validation inference completed!")

  0%|          | 0/1241 [00:00<?, ?it/s]

Prediction shape: (2481, 10)
Full validation inference completed!


In [24]:
from pathlib import Path
import pandas as pd

output_dir = ROOT / "reports/gate3_alignment"
output_dir.mkdir(parents=True, exist_ok=True)

probability_df = pd.DataFrame(
    probabilities,
    columns=categories
)

probability_df.insert(
    0,
    "chunk_id",
    val_chunks["chunk_id"].to_numpy()
)

output_path = output_dir / "candidate_a_probabilities.csv"

probability_df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", probability_df.shape)

Saved to: /content/Accenture-1D-contract-review-challenge/reports/gate3_alignment/candidate_a_probabilities.csv
Shape: (2481, 11)


In [25]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [26]:
print("Model loaded:", "model" in globals())
print("Validation data loaded:", "val_chunks" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())

Model loaded: True
Validation data loaded: True
Tokenizer loaded: True


In [27]:
import torch

device = torch.device("cuda")

model = model.to(device)
model.eval()

print("Model device:", next(model.parameters()).device)

Model device: cuda:0


In [28]:
import numpy as np
import torch
from tqdm.auto import tqdm

model.eval()

all_probabilities = []
batch_size = 2

for start in tqdm(range(0, len(val_chunks), batch_size)):

    batch_texts = (
        val_chunks["text"]
        .iloc[start:start + batch_size]
        .fillna("")
        .tolist()
    )

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=4096,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)

    all_probabilities.append(
        probs.cpu().numpy()
    )

probabilities = np.concatenate(
    all_probabilities,
    axis=0
)

print("Inference complete!")
print("Prediction shape:", probabilities.shape)

assert probabilities.shape == (2481, 10)

  0%|          | 0/1241 [00:00<?, ?it/s]

Inference complete!
Prediction shape: (2481, 10)


In [29]:
from pathlib import Path
import pandas as pd

output_dir = ROOT / "reports/gate3_alignment"
output_dir.mkdir(parents=True, exist_ok=True)

probability_df = pd.DataFrame(
    probabilities,
    columns=categories
)

probability_df.insert(
    0,
    "chunk_id",
    val_chunks["chunk_id"].to_numpy()
)

output_path = output_dir / "candidate_a_probabilities.csv"

probability_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", probability_df.shape)

Saved: /content/Accenture-1D-contract-review-challenge/reports/gate3_alignment/candidate_a_probabilities.csv
Shape: (2481, 11)


In [30]:
import numpy as np

threshold_array = np.array([
    thresholds_data[category]
    for category in categories
])

predictions = (
    probabilities >= threshold_array[None, :]
).astype(int)

print("Prediction shape:", predictions.shape)
print("Total positive predictions:", predictions.sum())

Prediction shape: (2481, 10)
Total positive predictions: 1338


In [31]:
import pandas as pd

# Convert probabilities into long format
long_probs = probability_df.melt(
    id_vars="chunk_id",
    var_name="category",
    value_name="probability"
)

# Match the frozen validation evaluation pairs
results = val_labels.merge(
    long_probs,
    on=["chunk_id", "category"],
    how="left",
    validate="one_to_one"
)

# Verify every evaluation pair has a prediction
assert len(results) == 24810
assert results["probability"].notna().all()

# Apply category-specific thresholds
results["threshold"] = results["category"].map(
    thresholds_data
)

assert results["threshold"].notna().all()

results["prediction"] = (
    results["probability"] >= results["threshold"]
).astype(int)

print("Final shape:", results.shape)
display(results.head())

Final shape: (24810, 11)


,chunk_id,title,config,chunk_index,category,target,mask,label,probability,threshold,prediction
0,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,0,Anti-Assignment,0,0,negative,0.000228,0.405786,0
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,0,Change Of Control,0,0,negative,0.000088,0.315231,0
2,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,0,Expiration Date,0,0,negative,0.000542,0.027410,0
3,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,0,Renewal Term,0,0,negative,0.000206,0.028887,0
4,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",768_512,0,License Grant,0,0,negative,0.001151,0.414239,0


In [32]:
print("Mask values:")
print(results["mask"].value_counts(dropna=False))

print("\nTarget by mask:")
print(pd.crosstab(
    results["mask"],
    results["target"]
))

print("\nLabel by mask:")
print(pd.crosstab(
    results["mask"],
    results["label"]
))

Mask values:
mask
0    24646
1      164
Name: count, dtype: int64

Target by mask:
target      0     1
mask               
0       23304  1342
1         164     0

Label by mask:
label  masked  negative  positive
mask                             
0           0     23304      1342
1         164         0         0


In [33]:
final_path = (
    output_dir /
    "transformer_validation_predictions.csv"
)

results.to_csv(final_path, index=False)

print("Saved:", final_path)
print("Rows:", len(results))
print("Columns:", results.columns.tolist())

Saved: /content/Accenture-1D-contract-review-challenge/reports/gate3_alignment/transformer_validation_predictions.csv
Rows: 24810
Columns: ['chunk_id', 'title', 'config', 'chunk_index', 'category', 'target', 'mask', 'label', 'probability', 'threshold', 'prediction']


In [34]:
print("Mask distribution:")
print(results["mask"].value_counts(dropna=False))

print("\nTarget by mask:")
print(pd.crosstab(
    results["mask"],
    results["target"]
))

print("\nLabel by mask:")
print(pd.crosstab(
    results["mask"],
    results["label"]
))

print("\nPrediction distribution:")
print(results["prediction"].value_counts())

Mask distribution:
mask
0    24646
1      164
Name: count, dtype: int64

Target by mask:
target      0     1
mask               
0       23304  1342
1         164     0

Label by mask:
label  masked  negative  positive
mask                             
0           0     23304      1342
1         164         0         0

Prediction distribution:
prediction
0    23472
1     1338
Name: count, dtype: int64


In [35]:
assert len(results) == 24810

assert not results.duplicated(
    ["chunk_id", "category"]
).any()

assert results["probability"].notna().all()
assert results["threshold"].notna().all()

assert results["probability"].between(0, 1).all()
assert results["prediction"].isin([0, 1]).all()

print("All structural validation checks passed!")

All structural validation checks passed!


In [36]:
import shutil

source = (
    ROOT /
    "reports/gate3_alignment/"
    "transformer_validation_predictions.csv"
)

destination = (
    "/content/drive/MyDrive/"
    "transformer_validation_predictions.csv"
)

shutil.copy2(source, destination)

print("Backup saved:", destination)

Backup saved: /content/drive/MyDrive/transformer_validation_predictions.csv


In [37]:
print("Mask distribution:")
print(results["mask"].value_counts(dropna=False))

print("\nTarget by mask:")
print(pd.crosstab(
    results["mask"],
    results["target"]
))

print("\nLabel by mask:")
print(pd.crosstab(
    results["mask"],
    results["label"]
))

print("\nPrediction distribution:")
print(results["prediction"].value_counts())

Mask distribution:
mask
0    24646
1      164
Name: count, dtype: int64

Target by mask:
target      0     1
mask               
0       23304  1342
1         164     0

Label by mask:
label  masked  negative  positive
mask                             
0           0     23304      1342
1         164         0         0

Prediction distribution:
prediction
0    23472
1     1338
Name: count, dtype: int64


In [38]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Exclude masked rows
eval_results = results[results["mask"] == 0].copy()

y_true = eval_results["target"].astype(int)
y_pred = eval_results["prediction"].astype(int)

# Overall metrics
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
micro_f1 = f1_score(y_true, y_pred, average="binary", zero_division=0)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

# Macro-F1 across the 10 contract categories
category_f1 = []

for category in categories:
    subset = eval_results[
        eval_results["category"] == category
    ]

    score = f1_score(
        subset["target"],
        subset["prediction"],
        zero_division=0
    )

    category_f1.append(score)

macro_f1 = sum(category_f1) / len(category_f1)

print("Candidate A — Reproduced Evaluation")
print("----------------------------------")
print(f"Precision: {precision:.6f}")
print(f"Recall:    {recall:.6f}")
print(f"Micro-F1:  {micro_f1:.6f}")
print(f"Macro-F1:  {macro_f1:.6f}")

print("\nConfusion matrix:")
print(f"TP: {tp}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TN: {tn}")

Candidate A — Reproduced Evaluation
----------------------------------
Precision: 0.832543
Recall:    0.785395
Micro-F1:  0.808282
Macro-F1:  0.763764

Confusion matrix:
TP: 1054
FP: 212
FN: 288
TN: 23092


In [39]:
original = {
    "precision": 0.8325434439178515,
    "recall": 0.7853949329359166,
    "micro_f1": 0.808282208588957,
    "macro_f1": 0.7637635286636082
}

reproduced = {
    "precision": precision,
    "recall": recall,
    "micro_f1": micro_f1,
    "macro_f1": macro_f1
}

print("Metric comparison")
print("-----------------")

for metric in original:
    difference = reproduced[metric] - original[metric]

    print(
        f"{metric}: "
        f"Original={original[metric]:.6f}, "
        f"Reproduced={reproduced[metric]:.6f}, "
        f"Difference={difference:+.6f}"
    )

Metric comparison
-----------------
precision: Original=0.832543, Reproduced=0.832543, Difference=+0.000000
recall: Original=0.785395, Reproduced=0.785395, Difference=+0.000000
micro_f1: Original=0.808282, Reproduced=0.808282, Difference=+0.000000
macro_f1: Original=0.763764, Reproduced=0.763764, Difference=+0.000000


In [40]:
import json

metrics = {
    "model": "Candidate A - Unweighted ModernBERT",
    "validation_chunks": 2481,
    "evaluation_pairs": 24646,
    "masked_pairs": 164,
    "precision": float(precision),
    "recall": float(recall),
    "micro_f1": float(micro_f1),
    "macro_f1": float(macro_f1),
    "tp": int(tp),
    "fp": int(fp),
    "fn": int(fn),
    "tn": int(tn)
}

metrics_path = output_dir / "candidate_a_verified_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", metrics_path)

Saved: /content/Accenture-1D-contract-review-challenge/reports/gate3_alignment/candidate_a_verified_metrics.json


In [41]:
import json
import hashlib

checkpoint = "/content/drive/MyDrive/model.safetensors"

with open(candidate_dir / "file_hashes.json") as f:
    recorded_hashes = json.load(f)

h = hashlib.sha256()

with open(checkpoint, "rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        h.update(block)

print("Actual SHA256:", h.hexdigest())
print("\nRecorded hash information:")
print(json.dumps(recorded_hashes, indent=2))

Actual SHA256: cc8ff6a049fe46ccfde88992e758d7dae7f06b6871c4f327cf8606e72e27aa52

Recorded hash information:
{
  "category_order.json": "38e598b7053d693d87a7ce6988c7e61d7109cd44a4b1d044967b3a7d36180480",
  "config.json": "8e9c74478ae1457d41054cecf76e139da790e1d7073085f2d4bfa018eb70c730",
  "metadata.json": "b54e632c6f30ae719077ad7f6ee790095878a5729203b2b8a72dc870cc82dd57",
  "model.safetensors": "cc8ff6a049fe46ccfde88992e758d7dae7f06b6871c4f327cf8606e72e27aa52",
  "thresholds.json": "e1c789b2ae4eaed09475cbf9491920c373e50ddee78180d82fbc9a356738c2b5",
  "tokenizer.json": "0fc88ec706f80b2eb15174d04fa7377948ca3f07065985deec1e28d3fb1cda9d",
  "tokenizer_config.json": "fb54f027372062b2ca52282efb04d178a8b57167a00cd8f4e816515823a2c016"
}


In [42]:
import shutil

shutil.copy2(
    metrics_path,
    "/content/drive/MyDrive/candidate_a_verified_metrics.json"
)

print("Metrics backed up successfully!")

Metrics backed up successfully!


In [43]:
from pathlib import Path

print("September baseline-related files:\n")

for file in ROOT.rglob("*"):
    if file.is_file() and any(
        term in str(file).lower()
        for term in [
            "keyword",
            "tfidf",
            "tf_idf",
            "baseline",
            "alignment"
        ]
    ):
        print(file.relative_to(ROOT))

September baseline-related files:

notebooks/keyword_baseline.ipynb
notebooks/tf_idf_baseline.ipynb
configs/keyword_rules_v1.json
configs/keyword_thresholds_v1.json
configs/keyword_freeze_v1.md
reports/tf_idf_baseline/failures.csv
reports/tf_idf_baseline/config_summary.csv
reports/tf_idf_baseline/selected_model.npz
reports/tf_idf_baseline/validation_margins.csv
reports/tf_idf_baseline/category_results.csv
reports/tf_idf_baseline/selected_model.json
reports/tf_idf_baseline/threshold_searches.csv
reports/gate3_alignment/candidate_a_verified_metrics.json
reports/gate3_alignment/candidate_a_probabilities.csv
reports/gate3_alignment/transformer_validation_predictions.csv
reports/keyword_baseline/validation_metrics.csv
reports/keyword_baseline/validation_threshold_search.csv
reports/keyword_baseline/weak_behavior.md
reports/keyword_baseline/validation_predictions.csv
reports/keyword_baseline/train_sanity_metrics.csv
reports/keyword_baseline/validation_match_evidence.csv
reports/keyword_basel

In [44]:
import pandas as pd
import json

keyword_dir = ROOT / "reports/keyword_baseline"
tfidf_dir = ROOT / "reports/tf_idf_baseline"

files = {
    "Keyword predictions": keyword_dir / "validation_predictions.csv",
    "Keyword metrics": keyword_dir / "validation_metrics.csv",
    "TF-IDF margins": tfidf_dir / "validation_margins.csv",
    "TF-IDF category results": tfidf_dir / "category_results.csv",
}

for name, path in files.items():
    df = pd.read_csv(path)

    print(f"\n{'=' * 55}")
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("First 3 rows:")
    display(df.head(3))


Keyword predictions
Shape: (24810, 7)
Columns: ['chunk_id', 'title', 'category', 'rule_set_version', 'score', 'matched_groups', 'n_vetoed_matches']
First 3 rows:


,chunk_id,title,category,rule_set_version,score,matched_groups,n_vetoed_matches
0,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Anti-Assignment,keyword_rules_v1,0,NaN,0
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Cap On Liability,keyword_rules_v1,0,NaN,0
2,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Change Of Control,keyword_rules_v1,0,NaN,0



Keyword metrics
Shape: (10, 14)
Columns: ['category', 'threshold', 'max_possible_score', 'tp', 'fp', 'fn', 'tn', 'support', 'positive_contracts', 'predicted_positive', 'precision', 'recall', 'f1', 'low_support_warning']
First 3 rows:


,category,threshold,max_possible_score,tp,fp,fn,tn,support,positive_contracts,predicted_positive,precision,recall,f1,low_support_warning
0,Renewal Term,1,6,55,14,23,2381,78,31,69,0.7971,0.7051,0.7483,False
1,Expiration Date,3,11,115,22,77,2251,192,71,137,0.8394,0.5990,0.6991,False
2,License Grant,3,6,145,27,62,2224,207,41,172,0.8430,0.7005,0.7652,False



TF-IDF margins
Shape: (147876, 5)
Columns: ['config', 'category', 'chunk_id', 'target', 'margin']
First 3 rows:


,config,category,chunk_id,target,margin
0,"min_df=2, C=0.1",Anti-Assignment,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",0,-0.855368
1,"min_df=2, C=0.1",Anti-Assignment,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",0,-0.850721
2,"min_df=2, C=0.1",Anti-Assignment,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",0,-0.888857



TF-IDF category results
Shape: (60, 17)
Columns: ['config', 'min_df', 'C', 'category', 'n_iter', 'support', 'n_val_rows', 'threshold', 'tp', 'fp', 'fn', 'tn', 'predicted_positive', 'precision', 'recall', 'f1', 'all_positive']
First 3 rows:


,config,min_df,C,category,n_iter,support,n_val_rows,threshold,tp,fp,fn,tn,predicted_positive,precision,recall,f1,all_positive
0,"min_df=2, C=0.1",2,0.1,Anti-Assignment,24,217,2448,0.286146,156,16,61,2215,172,0.906977,0.718894,0.802057,False
1,"min_df=2, C=0.1",2,0.1,Cap On Liability,25,229,2446,-0.035610,160,45,69,2172,205,0.780488,0.698690,0.737327,False
2,"min_df=2, C=0.1",2,0.1,Change Of Control,43,73,2472,-0.076581,41,48,32,2351,89,0.460674,0.561644,0.506173,False


In [45]:
path = tfidf_dir / "selected_model.json"

with open(path) as f:
    tfidf_config = json.load(f)

print(json.dumps(tfidf_config, indent=2)[:6000])

{
  "selected_config": "min_df=2, C=10.0",
  "text_normalization": "text_normalization.normalize_text: NFKC, casefold, quote/dash equivalents, whitespace runs -> one space",
  "vectorizer": {
    "lowercase": false,
    "max_df": 1.0,
    "max_features": null,
    "min_df": 2,
    "ngram_range": [
      1,
      2
    ],
    "norm": "l2",
    "smooth_idf": true,
    "stop_words": null,
    "sublinear_tf": true,
    "token_pattern": "(?u)\\b\\w+\\b",
    "use_idf": true
  },
  "classifier": {
    "C": 10.0,
    "class_weight": "balanced",
    "dual": true,
    "fit_intercept": true,
    "loss": "squared_hinge",
    "max_iter": 10000,
    "penalty": "l2",
    "random_state": 20260902,
    "tol": 0.0001
  },
  "scoring": "margin[c] = tfidf(normalize_text(text)) @ coef[c] + intercept[c]; detected if margin[c] >= threshold[c]",
  "n_iter": {
    "Anti-Assignment": 362,
    "Cap On Liability": 191,
    "Change Of Control": 181,
    "Exclusivity": 367,
    "Expiration Date": 860,
    "License

In [46]:
import pandas as pd

keyword = pd.read_csv(
    ROOT / "reports/keyword_baseline/validation_predictions.csv"
)

tfidf = pd.read_csv(
    ROOT / "reports/tf_idf_baseline/validation_margins.csv"
)

transformer = pd.read_csv(
    ROOT / "reports/gate3_alignment/transformer_validation_predictions.csv"
)

selected_config = tfidf_config["selected_config"]

tfidf_selected = tfidf[
    tfidf["config"] == selected_config
].copy()

print("Selected TF-IDF config:", selected_config)

print("\nRow counts:")
print("Keyword:", len(keyword))
print("TF-IDF selected:", len(tfidf_selected))
print("Transformer:", len(transformer))

print("\nUnique chunk-category pairs:")
for name, df in [
    ("Keyword", keyword),
    ("TF-IDF", tfidf_selected),
    ("Transformer", transformer)
]:
    print(
        name,
        df[["chunk_id", "category"]]
        .drop_duplicates()
        .shape[0]
    )

print("\nDuplicate pairs:")
for name, df in [
    ("Keyword", keyword),
    ("TF-IDF", tfidf_selected),
    ("Transformer", transformer)
]:
    print(
        name,
        df.duplicated(["chunk_id", "category"]).sum()
    )

Selected TF-IDF config: min_df=2, C=10.0

Row counts:
Keyword: 24810
TF-IDF selected: 24646
Transformer: 24810

Unique chunk-category pairs:
Keyword 24810
TF-IDF 24646
Transformer 24810

Duplicate pairs:
Keyword 0
TF-IDF 0
Transformer 0


In [47]:
keys = ["chunk_id", "category"]

transformer_keys = pd.MultiIndex.from_frame(
    transformer[keys]
)

keyword_keys = pd.MultiIndex.from_frame(
    keyword[keys]
)

tfidf_keys = pd.MultiIndex.from_frame(
    tfidf_selected[keys]
)

print("Keyword keys match:",
      transformer_keys.equals(keyword_keys))

print("TF-IDF keys match:",
      transformer_keys.equals(tfidf_keys))

print("\nSet comparison:")

print(
    "Keyword missing pairs:",
    len(transformer_keys.difference(keyword_keys))
)

print(
    "TF-IDF missing pairs:",
    len(transformer_keys.difference(tfidf_keys))
)

print(
    "Keyword extra pairs:",
    len(keyword_keys.difference(transformer_keys))
)

print(
    "TF-IDF extra pairs:",
    len(tfidf_keys.difference(transformer_keys))
)

Keyword keys match: False
TF-IDF keys match: False

Set comparison:
Keyword missing pairs: 0
TF-IDF missing pairs: 164
Keyword extra pairs: 0
TF-IDF extra pairs: 0


In [48]:
keys = ["chunk_id", "category"]

# Identify the 164 missing TF-IDF pairs
missing_keys = (
    transformer[keys]
    .merge(
        tfidf_selected[keys],
        on=keys,
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    [keys]
)

# Look up their evaluation masks
missing_details = missing_keys.merge(
    transformer[keys + ["mask", "target", "label"]],
    on=keys,
    validate="one_to_one"
)

print("Missing TF-IDF pairs:", len(missing_details))

print("\nMask values of missing pairs:")
print(missing_details["mask"].value_counts())

print("\nLabels of missing pairs:")
print(missing_details["label"].value_counts())

assert len(missing_details) == 164
assert (missing_details["mask"] == 1).all()

print("\nPASS: TF-IDF excludes exactly the masked pairs!")

Missing TF-IDF pairs: 164

Mask values of missing pairs:
mask
1    164
Name: count, dtype: int64

Labels of missing pairs:
label
masked    164
Name: count, dtype: int64

PASS: TF-IDF excludes exactly the masked pairs!


In [49]:
aligned_targets = tfidf_selected.merge(
    transformer[
        ["chunk_id", "category", "target", "mask"]
    ],
    on=["chunk_id", "category"],
    suffixes=("_tfidf", "_transformer"),
    validate="one_to_one"
)

print("Aligned rows:", len(aligned_targets))

print(
    "Target mismatches:",
    (
        aligned_targets["target_tfidf"]
        != aligned_targets["target_transformer"]
    ).sum()
)

print(
    "Masked rows in TF-IDF:",
    (aligned_targets["mask"] == 1).sum()
)

assert len(aligned_targets) == 24646
assert (
    aligned_targets["target_tfidf"]
    == aligned_targets["target_transformer"]
).all()

print("\nPASS: Ground-truth labels match!")

Aligned rows: 24646
Target mismatches: 0
Masked rows in TF-IDF: 0

PASS: Ground-truth labels match!


In [50]:
import hashlib

model_path = (
    ROOT / "reports/tf_idf_baseline/selected_model.npz"
)

h = hashlib.sha256()

with open(model_path, "rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        h.update(block)

actual_hash = h.hexdigest()
expected_hash = tfidf_config["npz_file_sha256"]

print("Hash matches:", actual_hash == expected_hash)

assert actual_hash == expected_hash

Hash matches: True


In [51]:
keyword_thresholds = pd.read_csv(
    ROOT / "reports/keyword_baseline/validation_metrics.csv"
)[["category", "threshold"]]

keyword_aligned = keyword.merge(
    transformer[
        ["chunk_id", "category", "target", "mask"]
    ],
    on=["chunk_id", "category"],
    how="inner",
    validate="one_to_one"
)

keyword_aligned = keyword_aligned.merge(
    keyword_thresholds,
    on="category",
    how="left",
    validate="many_to_one"
)

assert len(keyword_aligned) == 24810
assert keyword_aligned["threshold"].notna().all()

keyword_aligned["prediction"] = (
    keyword_aligned["score"] >= keyword_aligned["threshold"]
).astype(int)

keyword_eval = keyword_aligned[
    keyword_aligned["mask"] == 0
].copy()

assert len(keyword_eval) == 24646

print("Keyword evaluation rows:", len(keyword_eval))
print("Keyword predicted positives:", keyword_eval["prediction"].sum())

Keyword evaluation rows: 24646
Keyword predicted positives: 1245


In [52]:
tfidf_results = pd.read_csv(
    ROOT / "reports/tf_idf_baseline/category_results.csv"
)

tfidf_thresholds = tfidf_results[
    tfidf_results["config"] == selected_config
][["category", "threshold"]].copy()

assert len(tfidf_thresholds) == 10
assert tfidf_thresholds["category"].is_unique

tfidf_eval = tfidf_selected.merge(
    tfidf_thresholds,
    on="category",
    how="left",
    validate="many_to_one"
)

assert len(tfidf_eval) == 24646
assert tfidf_eval["threshold"].notna().all()

tfidf_eval["prediction"] = (
    tfidf_eval["margin"] >= tfidf_eval["threshold"]
).astype(int)

print("TF-IDF evaluation rows:", len(tfidf_eval))
print("TF-IDF predicted positives:", tfidf_eval["prediction"].sum())

TF-IDF evaluation rows: 24646
TF-IDF predicted positives: 1295


In [53]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def evaluate_baseline(df, model_name):
    y_true = df["target"].astype(int)
    y_pred = df["prediction"].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    category_scores = []

    for category in categories:
        subset = df[df["category"] == category]

        category_scores.append(
            f1_score(
                subset["target"],
                subset["prediction"],
                zero_division=0
            )
        )

    return {
        "Model": model_name,
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "Micro-F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "Macro-F1": sum(category_scores) / len(category_scores),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn)
    }


comparison = pd.DataFrame([
    evaluate_baseline(
        keyword_eval, "September Keyword"
    ),
    evaluate_baseline(
        tfidf_eval, "September TF-IDF"
    ),
    evaluate_baseline(
        transformer[transformer["mask"] == 0],
        "October ModernBERT Candidate A"
    )
])

display(comparison.round(4))

,Model,Precision,Recall,Micro-F1,Macro-F1,TP,FP,FN,TN
0,September Keyword,0.7173,0.6654,0.6904,0.6730,893,352,449,22952
1,September TF-IDF,0.7622,0.7355,0.7486,0.7117,987,308,355,22996
2,October ModernBERT Candidate A,0.8325,0.7854,0.8083,0.7638,1054,212,288,23092


In [54]:
comparison_path = (
    output_dir / "three_model_comparison.csv"
)

comparison.to_csv(comparison_path, index=False)

print("Saved:", comparison_path)

Saved: /content/Accenture-1D-contract-review-challenge/reports/gate3_alignment/three_model_comparison.csv


In [55]:
# Check keyword baseline against saved metrics
keyword_saved = pd.read_csv(
    ROOT / "reports/keyword_baseline/validation_metrics.csv"
)

tfidf_saved = pd.read_csv(
    ROOT / "reports/tf_idf_baseline/category_results.csv"
)

tfidf_saved = tfidf_saved[
    tfidf_saved["config"] == selected_config
]

def verify_counts(name, actual_df, saved_df):
    actual = (
        actual_df.groupby("category")
        .apply(
            lambda g: pd.Series({
                "tp": ((g.target == 1) & (g.prediction == 1)).sum(),
                "fp": ((g.target == 0) & (g.prediction == 1)).sum(),
                "fn": ((g.target == 1) & (g.prediction == 0)).sum(),
                "tn": ((g.target == 0) & (g.prediction == 0)).sum()
            }),
            include_groups=False
        )
        .reset_index()
    )

    merged = actual.merge(
        saved_df[["category", "tp", "fp", "fn", "tn"]],
        on="category",
        suffixes=("_actual", "_saved"),
        validate="one_to_one"
    )

    assert len(merged) == 10

    for metric in ["tp", "fp", "fn", "tn"]:
        merged[f"{metric}_match"] = (
            merged[f"{metric}_actual"]
            == merged[f"{metric}_saved"]
        )

    match_columns = [
        "tp_match", "fp_match",
        "fn_match", "tn_match"
    ]

    print(name)
    print("All counts match:",
          merged[match_columns].all().all())

    display(merged[
        ["category"] + match_columns
    ])

    return merged[match_columns].all().all()


keyword_verified = verify_counts(
    "September Keyword",
    keyword_eval,
    keyword_saved
)

tfidf_verified = verify_counts(
    "September TF-IDF",
    tfidf_eval,
    tfidf_saved
)

September Keyword
All counts match: True


,category,tp_match,fp_match,fn_match,tn_match
0,Anti-Assignment,True,True,True,True
1,Cap On Liability,True,True,True,True
2,Change Of Control,True,True,True,True
3,Exclusivity,True,True,True,True
4,Expiration Date,True,True,True,True
5,License Grant,True,True,True,True
6,Non-Compete,True,True,True,True
7,Non-Transferable License,True,True,True,True
8,Renewal Term,True,True,True,True
9,Uncapped Liability,True,True,True,True


September TF-IDF
All counts match: True


,category,tp_match,fp_match,fn_match,tn_match
0,Anti-Assignment,True,True,True,True
1,Cap On Liability,True,True,True,True
2,Change Of Control,True,True,True,True
3,Exclusivity,True,True,True,True
4,Expiration Date,True,True,True,True
5,License Grant,True,True,True,True
6,Non-Compete,True,True,True,True
7,Non-Transferable License,True,True,True,True
8,Renewal Term,True,True,True,True
9,Uncapped Liability,True,True,True,True


In [56]:
import shutil

assert keyword_verified and tfidf_verified

comparison_path = output_dir / "three_model_comparison.csv"

comparison.to_csv(comparison_path, index=False)

shutil.copy2(
    comparison_path,
    "/content/drive/MyDrive/three_model_comparison.csv"
)

print("Comparison saved and backed up!")

Comparison saved and backed up!


In [57]:
from pathlib import Path
import json
import shutil

output_dir = ROOT / "reports/gate3_alignment"
output_dir.mkdir(parents=True, exist_ok=True)

assert keyword_verified and tfidf_verified

comparison.to_csv(
    output_dir / "three_model_comparison.csv",
    index=False
)

verification = {
    "status": "PASS",
    "evaluation_pairs": 24646,
    "excluded_masked_pairs": 164,
    "ground_truth_mismatches": 0,
    "keyword_counts_match": True,
    "tfidf_counts_match": True,
    "tfidf_artifact_hash_match": True,
    "modernbert_metrics_reproduced": True,
    "selected_tfidf_config": selected_config
}

with open(
    output_dir / "alignment_verification.json", "w"
) as f:
    json.dump(verification, f, indent=2)

for filename in [
    "three_model_comparison.csv",
    "alignment_verification.json"
]:
    shutil.copy2(
        output_dir / filename,
        Path("/content/drive/MyDrive") / filename
    )

print("Gate 3.1 verification complete!")
print("Results saved and backed up to Google Drive.")

Gate 3.1 verification complete!
Results saved and backed up to Google Drive.
